# DINOv2 vs DINOv3: Wafer Map Defect Classification Comparison

This notebook compares Meta's DINOv2 and DINOv3 vision foundation models as frozen feature extractors for semiconductor wafer bin map (WBM) defect classification.

**What we're testing:**
- Which backbone produces better representations for wafer defect patterns?
- How do they compare on few-shot classification (minimal labeled data)?
- How do the learned feature spaces differ visually?
- Where does DINOv3's Gram anchoring advantage actually show up?

**Dataset:** WM-811K (811,457 wafer bin maps, ~172K labeled across 9 defect classes)

**Requirements:** GPU with 16+ GB VRAM recommended (24GB for comfortable headroom)

## 1. Setup & Installation

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install transformers timm
# !pip install scikit-learn matplotlib seaborn pandas numpy
# !pip install umap-learn
# !pip install kagglehub  # For WM-811K download

In [ ]:
import os
import pickle
import warnings
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.colors import ListedColormap

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    import umap
    HAS_UMAP = True
except ImportError:
    from sklearn.manifold import TSNE
    HAS_UMAP = False
    print("UMAP not found, falling back to t-SNE for dimensionality reduction")

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Dataset Loading — WM-811K

The WM-811K dataset contains 811,457 wafer bin maps collected from real semiconductor production. Each map is a 2D grid where each cell represents a die (pass/fail). Only ~172K maps have expert-assigned defect labels.

**Defect classes:** Center, Donut, Edge-Loc, Edge-Ring, Loc, Near-full, None, Random, Scratch

In [ ]:
# ============================================================
# Option A: Download from Kaggle (recommended)
# ============================================================
# Uncomment and run if you have kagglehub installed:
# import kagglehub
# path = kagglehub.dataset_download("qingyi/wm811k-wafer-map")
# DATA_PATH = Path(path)

# ============================================================
# Option B: Manual download
# ============================================================
# Download LSWMD.pkl from: https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map
# Place it in the same directory as this notebook

DATA_PATH = Path('.')  # Update this to your data location
PKL_FILE = DATA_PATH / 'LSWMD.pkl'

assert PKL_FILE.exists(), (
    f"Dataset not found at {PKL_FILE}. "
    f"Download LSWMD.pkl from https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map"
)

In [ ]:
# Load the dataset
print("Loading WM-811K dataset...")
df = pd.read_pickle(PKL_FILE)
print(f"Total wafer maps: {len(df):,}")
print(f"Columns: {list(df.columns)}")

# The 'failureType' column contains the label
# Filter to only labeled samples
df['failureLabel'] = df['failureType'].apply(
    lambda x: x[0][0] if isinstance(x, np.ndarray) and x.size > 0 else 'unlabeled'
)

# Define the 8 defect classes (excluding 'none' which means no defect pattern)
DEFECT_CLASSES = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring',
                  'Loc', 'Near-full', 'Random', 'Scratch', 'none']

df_labeled = df[df['failureLabel'].isin(DEFECT_CLASSES)].copy()
df_labeled = df_labeled.reset_index(drop=True)

print(f"\nLabeled wafer maps: {len(df_labeled):,}")
print(f"Unlabeled wafer maps: {len(df) - len(df_labeled):,}")
print(f"Label ratio: {len(df_labeled)/len(df)*100:.1f}%")
print(f"\nClass distribution:")
print(df_labeled['failureLabel'].value_counts().to_string())

### 2.1 Visualize Sample Wafer Maps by Defect Class

In [ ]:
# Custom colormap for wafer maps: background (gray), pass (white), fail (red)
wafer_cmap = ListedColormap(['#2d2d2d', '#e8e8e8', '#d32f2f'])

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
fig.suptitle('WM-811K Defect Pattern Examples', fontsize=18, fontweight='bold', y=0.98)

for idx, defect_class in enumerate(DEFECT_CLASSES):
    ax = axes[idx // 3, idx % 3]
    samples = df_labeled[df_labeled['failureLabel'] == defect_class]
    if len(samples) > 0:
        # Pick a clear example (not the first, which may be noisy)
        sample = samples.iloc[min(5, len(samples) - 1)]
        wafer_map = sample['waferMap']
        ax.imshow(wafer_map, cmap=wafer_cmap, interpolation='nearest', vmin=0, vmax=2)
    ax.set_title(f"{defect_class}\n(n={len(samples):,})", fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('wafer_map_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Each wafer map shows a 2D grid of dies: dark gray = outside wafer,")
print("light gray = passing dies, red = failing dies. The spatial pattern of")
print("failures is what classifies the defect type.")

In [ ]:
# Class distribution bar chart
class_counts = df_labeled['failureLabel'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('viridis', len(class_counts))
bars = ax.bar(class_counts.index, class_counts.values, color=colors, edgecolor='white', linewidth=0.5)

for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{count:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('WM-811K Class Distribution (Labeled Subset)', fontsize=14, fontweight='bold')
ax.set_xlabel('Defect Class', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_yscale('log')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nClass imbalance ratio (max/min): {class_counts.max()/class_counts.min():.0f}x")
print("Note the severe class imbalance — 'none' and 'Loc' dominate while")
print("'Donut' and 'Near-full' are extremely rare. This mirrors real fab conditions.")

## 3. Data Preprocessing

Both DINOv2 and DINOv3 expect standard RGB images. We need to:
1. Resize variable-size wafer maps to a fixed resolution
2. Convert the 3-value grid (0=background, 1=pass, 2=fail) to RGB
3. Normalize using ImageNet stats (since both models were pretrained on natural images)

In [ ]:
class WaferMapDataset(Dataset):
    """PyTorch dataset for WM-811K wafer bin maps.
    
    Converts raw wafer maps to 3-channel RGB images suitable for
    vision transformer input. Background, passing, and failing dies
    are mapped to distinct colors to preserve spatial defect patterns.
    """
    
    # Color mapping: [background, pass, fail] -> RGB
    COLOR_MAP = np.array([
        [45, 45, 45],      # background: dark gray
        [232, 232, 232],   # pass: light gray  
        [211, 47, 47],     # fail: red
    ], dtype=np.uint8)
    
    def __init__(self, wafer_maps, labels, label_to_idx, img_size=224, transform=None):
        self.wafer_maps = wafer_maps
        self.labels = labels
        self.label_to_idx = label_to_idx
        self.img_size = img_size
        self.transform = transform or self._default_transform()
    
    def _default_transform(self):
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                std=[0.229, 0.224, 0.225]
            ),
        ])
    
    def _wafer_to_rgb(self, wafer_map):
        """Convert 2D wafer map (values 0,1,2) to RGB image."""
        wafer_clipped = np.clip(wafer_map.astype(int), 0, 2)
        rgb = self.COLOR_MAP[wafer_clipped]  # H x W x 3
        return rgb
    
    def __len__(self):
        return len(self.wafer_maps)
    
    def __getitem__(self, idx):
        wafer_map = self.wafer_maps[idx]
        label = self.label_to_idx[self.labels[idx]]
        rgb = self._wafer_to_rgb(wafer_map)
        img = self.transform(rgb)
        return img, label


# Build label mapping
label_to_idx = {label: idx for idx, label in enumerate(DEFECT_CLASSES)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
NUM_CLASSES = len(DEFECT_CLASSES)

print(f"Label mapping: {label_to_idx}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# Train/test split with stratification to handle class imbalance
wafer_maps = df_labeled['waferMap'].values
labels = df_labeled['failureLabel'].values

train_maps, test_maps, train_labels, test_labels = train_test_split(
    wafer_maps, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training set: {len(train_maps):,} samples")
print(f"Test set:     {len(test_maps):,} samples")

# Create datasets
# DINOv2 uses 518px but 224px works for linear probing comparison
# Using 224px keeps memory manageable and is fair to both models
IMG_SIZE = 224
BATCH_SIZE = 64

train_dataset = WaferMapDataset(train_maps, train_labels, label_to_idx, img_size=IMG_SIZE)
test_dataset = WaferMapDataset(test_maps, test_labels, label_to_idx, img_size=IMG_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Verify a batch
sample_imgs, sample_labels = next(iter(train_loader))
print(f"\nBatch shape: {sample_imgs.shape}")
print(f"Label batch: {sample_labels[:10]}")

In [ ]:
# Visualize preprocessed wafer maps (after RGB conversion, before normalization)
vis_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle(f'Preprocessed Wafer Maps ({IMG_SIZE}x{IMG_SIZE} RGB)', fontsize=14, fontweight='bold')

for i in range(10):
    ax = axes[i // 5, i % 5]
    wafer_rgb = WaferMapDataset.COLOR_MAP[np.clip(train_maps[i].astype(int), 0, 2)]
    pil_img = vis_transform(wafer_rgb)
    ax.imshow(pil_img)
    ax.set_title(train_labels[i], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('preprocessed_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("These are the images both DINOv2 and DINOv3 will see as input.")

## 4. Load Models

We load both models as **frozen feature extractors** — no fine-tuning of the backbone.
This tests the raw quality of each model's pretrained representations.

- **DINOv2 ViT-L/14**: 304M params, pretrained on LVD-142M
- **DINOv3 ViT-L/14**: Distilled from 7B teacher, pretrained on 1.7B images

In [ ]:
# ============================================================
# Load DINOv2 ViT-L/14
# ============================================================
print("Loading DINOv2 ViT-L/14...")
dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
dinov2_model = dinov2_model.to(DEVICE)
dinov2_model.eval()

dinov2_embed_dim = dinov2_model.embed_dim
print(f"DINOv2 loaded — embedding dim: {dinov2_embed_dim}")
print(f"Parameters: {sum(p.numel() for p in dinov2_model.parameters()):,}")

In [ ]:
# ============================================================
# Load DINOv3 ViT-L/14
# ============================================================
# DINOv3 is available via torch.hub from Meta's repo
# If torch.hub doesn't work, use Hugging Face transformers as fallback

print("Loading DINOv3 ViT-L/14...")
try:
    dinov3_model = torch.hub.load('facebookresearch/dinov3', 'dinov3_vitl14')
    print("Loaded via torch.hub")
except Exception as e:
    print(f"torch.hub failed ({e}), trying Hugging Face...")
    try:
        from transformers import AutoModel
        dinov3_model = AutoModel.from_pretrained('facebook/dinov3-large')
        print("Loaded via Hugging Face transformers")
    except Exception as e2:
        print(f"HuggingFace also failed ({e2}).")
        print("Manual fallback: download weights from https://huggingface.co/facebook/dinov3-large")
        print("and load with: dinov3_model = torch.load('dinov3_vitl14.pth')")
        raise

dinov3_model = dinov3_model.to(DEVICE)
dinov3_model.eval()

dinov3_embed_dim = dinov3_model.embed_dim if hasattr(dinov3_model, 'embed_dim') else 1024
print(f"DINOv3 loaded — embedding dim: {dinov3_embed_dim}")
print(f"Parameters: {sum(p.numel() for p in dinov3_model.parameters()):,}")

## 5. Feature Extraction

Pass all wafer maps through both frozen backbones to extract CLS token embeddings.
These embeddings are what the linear probe will classify.

In [ ]:
@torch.no_grad()
def extract_features(model, dataloader, device, model_name='model'):
    """Extract CLS token features from a frozen DINO model.
    
    Returns:
        features: np.array of shape (N, embed_dim)
        labels: np.array of shape (N,)
        time_elapsed: float, seconds
    """
    all_features = []
    all_labels = []
    
    start = time.time()
    total_batches = len(dataloader)
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        
        # Both DINOv2 and DINOv3 output CLS token via forward()
        features = model(images)
        
        # Handle different output formats
        if isinstance(features, dict):
            # Some HuggingFace wrappers return dict
            features = features.get('last_hidden_state', features.get('pooler_output'))
            if features.dim() == 3:
                features = features[:, 0]  # CLS token
        
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())
        
        if (batch_idx + 1) % 50 == 0 or batch_idx == total_batches - 1:
            elapsed = time.time() - start
            print(f"  [{model_name}] Batch {batch_idx+1}/{total_batches} "
                  f"({elapsed:.1f}s elapsed)")
    
    elapsed = time.time() - start
    features = np.concatenate(all_features, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    
    print(f"  [{model_name}] Done — {features.shape[0]:,} samples, "
          f"{features.shape[1]}d embeddings, {elapsed:.1f}s total")
    return features, labels, elapsed

In [ ]:
print("="*60)
print("Extracting features with DINOv2...")
print("="*60)
train_feats_v2, train_labels_arr, v2_train_time = extract_features(
    dinov2_model, train_loader, DEVICE, 'DINOv2'
)
test_feats_v2, test_labels_arr, v2_test_time = extract_features(
    dinov2_model, test_loader, DEVICE, 'DINOv2'
)

print(f"\n{'='*60}")
print("Extracting features with DINOv3...")
print("="*60)
train_feats_v3, _, v3_train_time = extract_features(
    dinov3_model, train_loader, DEVICE, 'DINOv3'
)
test_feats_v3, _, v3_test_time = extract_features(
    dinov3_model, test_loader, DEVICE, 'DINOv3'
)

print(f"\n{'='*60}")
print("Feature Extraction Speed Comparison")
print(f"{'='*60}")
print(f"DINOv2: {v2_train_time + v2_test_time:.1f}s total")
print(f"DINOv3: {v3_train_time + v3_test_time:.1f}s total")
print(f"Speed ratio: {(v3_train_time + v3_test_time) / (v2_train_time + v2_test_time):.2f}x")

## 6. Embedding Space Visualization

Reduce the high-dimensional embeddings to 2D to visualize how each model separates the defect classes. Better separation in 2D = more linearly separable features = easier for a simple classifier.

In [ ]:
# Subsample for visualization (full dataset takes too long for dim reduction)
N_VIS = 5000
np.random.seed(42)
vis_idx = np.random.choice(len(test_feats_v2), min(N_VIS, len(test_feats_v2)), replace=False)

vis_feats_v2 = test_feats_v2[vis_idx]
vis_feats_v3 = test_feats_v3[vis_idx]
vis_labels = test_labels_arr[vis_idx]

# Normalize features before dim reduction
scaler_v2 = StandardScaler()
scaler_v3 = StandardScaler()
vis_feats_v2_norm = scaler_v2.fit_transform(vis_feats_v2)
vis_feats_v3_norm = scaler_v3.fit_transform(vis_feats_v3)

print(f"Running dimensionality reduction on {len(vis_idx):,} samples...")

if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    method_name = 'UMAP'
else:
    reducer = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    method_name = 't-SNE'

print(f"Using {method_name}...")
embed_v2 = reducer.fit_transform(vis_feats_v2_norm)

if HAS_UMAP:
    reducer2 = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
else:
    reducer2 = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embed_v3 = reducer2.fit_transform(vis_feats_v3_norm)

print("Done.")

In [ ]:
# Side-by-side embedding visualization
label_names = [idx_to_label[l] for l in vis_labels]
unique_labels = sorted(set(label_names))
palette = sns.color_palette('husl', len(unique_labels))
color_map = {label: palette[i] for i, label in enumerate(unique_labels)}
colors = [color_map[l] for l in label_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

for label in unique_labels:
    mask = np.array(label_names) == label
    ax1.scatter(embed_v2[mask, 0], embed_v2[mask, 1],
               c=[color_map[label]], label=label, s=8, alpha=0.6)
    ax2.scatter(embed_v3[mask, 0], embed_v3[mask, 1],
               c=[color_map[label]], label=label, s=8, alpha=0.6)

ax1.set_title(f'DINOv2 Feature Space ({method_name})', fontsize=14, fontweight='bold')
ax2.set_title(f'DINOv3 Feature Space ({method_name})', fontsize=14, fontweight='bold')

for ax in [ax1, ax2]:
    ax.legend(markerscale=3, fontsize=9, loc='best', framealpha=0.8)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Embedding Space Comparison: How Each Model Separates Defect Classes',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('embedding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Tighter, more separated clusters = better linear separability.")
print("Compare how cleanly each model groups the defect types.")

## 7. Linear Probe Classification

Train a simple logistic regression classifier on the frozen features from each model. This is the standard evaluation protocol — it tests how useful the representations are without any backbone fine-tuning.

In [ ]:
def train_and_evaluate_probe(train_feats, test_feats, train_labels, test_labels, model_name):
    """Train a logistic regression linear probe and return metrics."""
    # Normalize features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_feats)
    X_test = scaler.transform(test_feats)
    
    print(f"\nTraining linear probe on {model_name} features...")
    start = time.time()
    
    clf = LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver='lbfgs',
        multi_class='multinomial',
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train, train_labels)
    
    train_time = time.time() - start
    
    # Predictions
    train_pred = clf.predict(X_train)
    test_pred = clf.predict(X_test)
    test_proba = clf.predict_proba(X_test)
    
    # Metrics
    results = {
        'model': model_name,
        'train_acc': accuracy_score(train_labels, train_pred),
        'test_acc': accuracy_score(test_labels, test_pred),
        'test_f1_macro': f1_score(test_labels, test_pred, average='macro'),
        'test_f1_weighted': f1_score(test_labels, test_pred, average='weighted'),
        'train_time': train_time,
        'predictions': test_pred,
        'probabilities': test_proba,
        'classifier': clf,
        'scaler': scaler,
    }
    
    print(f"  Train acc: {results['train_acc']:.4f}")
    print(f"  Test acc:  {results['test_acc']:.4f}")
    print(f"  Test F1 (macro):    {results['test_f1_macro']:.4f}")
    print(f"  Test F1 (weighted): {results['test_f1_weighted']:.4f}")
    print(f"  Probe training time: {train_time:.2f}s")
    
    return results


# Run both
results_v2 = train_and_evaluate_probe(
    train_feats_v2, test_feats_v2, train_labels_arr, test_labels_arr, 'DINOv2'
)
results_v3 = train_and_evaluate_probe(
    train_feats_v3, test_feats_v3, train_labels_arr, test_labels_arr, 'DINOv3'
)

## 8. Results Visualization

### 8.1 Overall Metrics Comparison

In [ ]:
# Summary metrics bar chart
metrics = ['test_acc', 'test_f1_macro', 'test_f1_weighted']
metric_labels = ['Test Accuracy', 'F1 (Macro)', 'F1 (Weighted)']

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(metrics))
width = 0.3

v2_vals = [results_v2[m] for m in metrics]
v3_vals = [results_v3[m] for m in metrics]

bars1 = ax.bar(x - width/2, v2_vals, width, label='DINOv2 ViT-L',
               color='#4285F4', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + width/2, v3_vals, width, label='DINOv3 ViT-L',
               color='#EA4335', edgecolor='white', linewidth=0.5)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.003,
                f'{height:.3f}', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('DINOv2 vs DINOv3: Linear Probe on WM-811K',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.08)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add improvement annotation
for i, (v2, v3) in enumerate(zip(v2_vals, v3_vals)):
    diff = v3 - v2
    sign = '+' if diff > 0 else ''
    ax.annotate(f'{sign}{diff:.3f}',
                xy=(i + width/2, max(v2, v3) + 0.02),
                fontsize=9, color='green' if diff > 0 else 'red',
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('overall_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.2 Confusion Matrices

In [ ]:
# Side-by-side confusion matrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

cm_v2 = confusion_matrix(test_labels_arr, results_v2['predictions'])
cm_v3 = confusion_matrix(test_labels_arr, results_v3['predictions'])

# Normalize to percentages
cm_v2_pct = cm_v2.astype('float') / cm_v2.sum(axis=1)[:, np.newaxis] * 100
cm_v3_pct = cm_v3.astype('float') / cm_v3.sum(axis=1)[:, np.newaxis] * 100

sns.heatmap(cm_v2_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=DEFECT_CLASSES, yticklabels=DEFECT_CLASSES,
            ax=ax1, cbar_kws={'label': '% of true class'})
ax1.set_title(f'DINOv2 (Acc: {results_v2["test_acc"]:.3f})', fontsize=14, fontweight='bold')
ax1.set_xlabel('Predicted', fontsize=12)
ax1.set_ylabel('True', fontsize=12)

sns.heatmap(cm_v3_pct, annot=True, fmt='.1f', cmap='Reds',
            xticklabels=DEFECT_CLASSES, yticklabels=DEFECT_CLASSES,
            ax=ax2, cbar_kws={'label': '% of true class'})
ax2.set_title(f'DINOv3 (Acc: {results_v3["test_acc"]:.3f})', fontsize=14, fontweight='bold')
ax2.set_xlabel('Predicted', fontsize=12)
ax2.set_ylabel('True', fontsize=12)

plt.suptitle('Confusion Matrices (Normalized by Row)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Look for which classes each model confuses — especially rare classes")
print("like Donut and Near-full where both models are likely to struggle.")

### 8.3 Per-Class F1 Scores

In [ ]:
# Per-class F1 comparison
from sklearn.metrics import f1_score as f1_func

f1_per_class_v2 = f1_score(test_labels_arr, results_v2['predictions'],
                           labels=range(NUM_CLASSES), average=None)
f1_per_class_v3 = f1_score(test_labels_arr, results_v3['predictions'],
                           labels=range(NUM_CLASSES), average=None)

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(NUM_CLASSES)
width = 0.35

bars1 = ax.bar(x - width/2, f1_per_class_v2, width, label='DINOv2',
               color='#4285F4', edgecolor='white')
bars2 = ax.bar(x + width/2, f1_per_class_v3, width, label='DINOv3',
               color='#EA4335', edgecolor='white')

# Highlight improvements
for i, (v2, v3) in enumerate(zip(f1_per_class_v2, f1_per_class_v3)):
    diff = v3 - v2
    if abs(diff) > 0.005:
        sign = '+' if diff > 0 else ''
        ax.annotate(f'{sign}{diff:.3f}',
                    xy=(i, max(v2, v3) + 0.02),
                    fontsize=8, ha='center',
                    color='green' if diff > 0 else 'red',
                    fontweight='bold')

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1 Score Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(DEFECT_CLASSES, rotation=30, ha='right', fontsize=11)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.12)
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.3, label='90% threshold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key insight: watch the rare classes (Donut, Near-full).")
print("DINOv3's richer feature space should help most on these.")

### 8.4 Detailed Classification Report

In [ ]:
print("=" * 70)
print("DINOV2 — Classification Report")
print("=" * 70)
print(classification_report(
    test_labels_arr, results_v2['predictions'],
    target_names=DEFECT_CLASSES, digits=4
))

print("\n" + "=" * 70)
print("DINOV3 — Classification Report")
print("=" * 70)
print(classification_report(
    test_labels_arr, results_v3['predictions'],
    target_names=DEFECT_CLASSES, digits=4
))

## 9. Few-Shot Comparison

This is the most practically relevant experiment. In a real fab, labeled data is expensive. How many labeled examples per class does each model need to reach production-quality accuracy?

We train linear probes with increasing numbers of labeled samples per class and track accuracy.

In [ ]:
def few_shot_evaluation(train_feats, test_feats, train_labels, test_labels,
                        shots_list, model_name, n_trials=5):
    """Evaluate linear probe accuracy with varying numbers of labeled examples.
    
    Args:
        shots_list: list of ints, number of labeled examples per class
        n_trials: number of random trials to average over
    
    Returns:
        dict with accuracies and f1 scores for each shot count
    """
    results = {'shots': [], 'acc_mean': [], 'acc_std': [],
               'f1_mean': [], 'f1_std': []}
    
    scaler = StandardScaler()
    X_train_full = scaler.fit_transform(train_feats)
    X_test = scaler.transform(test_feats)
    
    unique_labels = np.unique(train_labels)
    
    for n_shots in shots_list:
        trial_accs = []
        trial_f1s = []
        
        for trial in range(n_trials):
            # Sample n_shots examples per class
            selected_idx = []
            rng = np.random.RandomState(trial * 1000 + n_shots)
            
            for label in unique_labels:
                label_idx = np.where(train_labels == label)[0]
                n_available = min(n_shots, len(label_idx))
                chosen = rng.choice(label_idx, n_available, replace=False)
                selected_idx.extend(chosen)
            
            X_sub = X_train_full[selected_idx]
            y_sub = train_labels[selected_idx]
            
            clf = LogisticRegression(
                max_iter=1000, C=1.0, solver='lbfgs',
                multi_class='multinomial', random_state=42
            )
            clf.fit(X_sub, y_sub)
            
            pred = clf.predict(X_test)
            trial_accs.append(accuracy_score(test_labels, pred))
            trial_f1s.append(f1_score(test_labels, pred, average='macro'))
        
        results['shots'].append(n_shots)
        results['acc_mean'].append(np.mean(trial_accs))
        results['acc_std'].append(np.std(trial_accs))
        results['f1_mean'].append(np.mean(trial_f1s))
        results['f1_std'].append(np.std(trial_f1s))
        
        print(f"  [{model_name}] {n_shots:>5} shots/class → "
              f"Acc: {np.mean(trial_accs):.4f} ± {np.std(trial_accs):.4f}, "
              f"F1: {np.mean(trial_f1s):.4f} ± {np.std(trial_f1s):.4f}")
    
    return results


# Define shot counts to test
SHOTS_LIST = [5, 10, 20, 50, 100, 200, 500, 1000]

print("Running few-shot evaluation (this may take a few minutes)...\n")

print("DINOv2:")
fewshot_v2 = few_shot_evaluation(
    train_feats_v2, test_feats_v2, train_labels_arr, test_labels_arr,
    SHOTS_LIST, 'DINOv2'
)

print("\nDINOv3:")
fewshot_v3 = few_shot_evaluation(
    train_feats_v3, test_feats_v3, train_labels_arr, test_labels_arr,
    SHOTS_LIST, 'DINOv3'
)

In [ ]:
# Few-shot learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Accuracy curve
ax1.errorbar(fewshot_v2['shots'], fewshot_v2['acc_mean'], yerr=fewshot_v2['acc_std'],
             marker='o', linewidth=2, capsize=4, label='DINOv2', color='#4285F4', markersize=8)
ax1.errorbar(fewshot_v3['shots'], fewshot_v3['acc_mean'], yerr=fewshot_v3['acc_std'],
             marker='s', linewidth=2, capsize=4, label='DINOv3', color='#EA4335', markersize=8)

ax1.set_xscale('log')
ax1.set_xlabel('Labeled Samples per Class', fontsize=12)
ax1.set_ylabel('Test Accuracy', fontsize=12)
ax1.set_title('Few-Shot Accuracy', fontsize=14, fontweight='bold')
ax1.legend(fontsize=12)
ax1.axhline(y=0.95, color='gray', linestyle='--', alpha=0.4)
ax1.text(SHOTS_LIST[0], 0.953, '95% threshold', fontsize=9, color='gray')
ax1.grid(True, alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# F1 Macro curve
ax2.errorbar(fewshot_v2['shots'], fewshot_v2['f1_mean'], yerr=fewshot_v2['f1_std'],
             marker='o', linewidth=2, capsize=4, label='DINOv2', color='#4285F4', markersize=8)
ax2.errorbar(fewshot_v3['shots'], fewshot_v3['f1_mean'], yerr=fewshot_v3['f1_std'],
             marker='s', linewidth=2, capsize=4, label='DINOv3', color='#EA4335', markersize=8)

ax2.set_xscale('log')
ax2.set_xlabel('Labeled Samples per Class', fontsize=12)
ax2.set_ylabel('Macro F1 Score', fontsize=12)
ax2.set_title('Few-Shot F1 (Macro-Averaged)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Few-Shot Learning Curves: How Many Labels Does Each Model Need?',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('few_shot_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey question: At what shot count does each model cross 95% accuracy?")
print("The model that gets there first needs fewer labeled examples — directly")
print("translating to less expert labeling effort in the fab.")

In [ ]:
# Delta plot: DINOv3 advantage at each shot count
acc_delta = np.array(fewshot_v3['acc_mean']) - np.array(fewshot_v2['acc_mean'])
f1_delta = np.array(fewshot_v3['f1_mean']) - np.array(fewshot_v2['f1_mean'])

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(SHOTS_LIST))
width = 0.35

bars1 = ax.bar(x - width/2, acc_delta * 100, width, label='Accuracy Δ',
               color=['#34A853' if d > 0 else '#EA4335' for d in acc_delta],
               edgecolor='white')
bars2 = ax.bar(x + width/2, f1_delta * 100, width, label='F1 Macro Δ',
               color=['#34A853' if d > 0 else '#FBBC04' for d in f1_delta],
               edgecolor='white', alpha=0.7)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_ylabel('DINOv3 − DINOv2 (percentage points)', fontsize=11)
ax.set_title('DINOv3 Advantage Over DINOv2 by Shot Count',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in SHOTS_LIST], fontsize=11)
ax.set_xlabel('Labeled Samples per Class', fontsize=12)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('dinov3_advantage_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print("Green bars above zero = DINOv3 wins at that shot count.")
print("Largest advantages expected at low shot counts (5-50) where")
print("richer representations matter most.")

## 10. Attention Map Visualization

Visualize where each model "looks" when processing wafer maps. This reveals whether the models attend to the defect patterns or to irrelevant structure.

In [ ]:
@torch.no_grad()
def get_attention_maps(model, img_tensor, device, patch_size=14):
    """Extract self-attention maps from the last layer of a ViT model.
    
    Returns:
        attn_map: np.array of shape (H_patches, W_patches)
    """
    img = img_tensor.unsqueeze(0).to(device)
    
    # Get intermediate attention weights
    # DINOv2/v3 via torch.hub expose get_intermediate_layers
    try:
        # Method 1: Use built-in attention extraction
        output = model.get_intermediate_layers(
            img, n=1, return_class_token=True,
            reshape=True
        )
        # Use the patch tokens from last layer
        patch_tokens = output[0][0]  # (1, H, W, C)
        if patch_tokens.dim() == 4:
            # Compute attention as norm of patch tokens
            attn = patch_tokens.squeeze(0).norm(dim=-1)  # H x W
        else:
            attn = patch_tokens.squeeze(0).norm(dim=-1)
        attn = attn.cpu().numpy()
    except Exception:
        try:
            # Method 2: Register forward hooks on attention layers
            attention_weights = []
            
            def hook_fn(module, input, output):
                if isinstance(output, tuple):
                    attention_weights.append(output[1])  # attention weights
            
            # Try to find the last attention layer
            hooks = []
            for name, module in model.named_modules():
                if 'attn' in name.lower() and hasattr(module, 'qkv'):
                    hooks.append(module.register_forward_hook(hook_fn))
            
            _ = model(img)
            for h in hooks:
                h.remove()
            
            if attention_weights:
                # Average over heads, get CLS->patch attention
                last_attn = attention_weights[-1]
                cls_attn = last_attn[0, :, 0, 1:]  # heads x patches
                cls_attn = cls_attn.mean(0)  # average over heads
                h = w = int(cls_attn.shape[0] ** 0.5)
                attn = cls_attn.reshape(h, w).cpu().numpy()
            else:
                raise RuntimeError("No attention weights captured")
        except Exception:
            # Method 3: Fallback — use feature norms as proxy
            output = model.forward_features(img) if hasattr(model, 'forward_features') else model(img)
            if isinstance(output, dict):
                tokens = output.get('x_norm_patchtokens', output.get('last_hidden_state'))
            else:
                tokens = output
            if tokens.dim() == 3:
                patch_tokens = tokens[0, 1:]  # skip CLS
            else:
                patch_tokens = tokens[0]
            h = w = int(patch_tokens.shape[0] ** 0.5)
            attn = patch_tokens.norm(dim=-1).reshape(h, w).cpu().numpy()
    
    # Normalize to [0, 1]
    attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
    return attn

print("Attention extraction function ready.")

In [ ]:
# Select interesting examples from each class for attention visualization
vis_classes = ['Center', 'Scratch', 'Edge-Ring', 'Donut', 'Random', 'Loc']
n_examples = len(vis_classes)

fig, axes = plt.subplots(n_examples, 4, figsize=(16, 4 * n_examples))

vis_unnorm = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

for row, defect_class in enumerate(vis_classes):
    # Find an example of this class
    class_idx = label_to_idx[defect_class]
    mask = test_labels_arr == class_idx
    sample_idx = np.where(mask)[0][0]
    
    # Get the image tensor
    img_tensor, _ = test_dataset[sample_idx]
    
    # Get the raw wafer map for display
    wafer_rgb = WaferMapDataset.COLOR_MAP[
        np.clip(test_maps[sample_idx].astype(int), 0, 2)
    ]
    raw_img = vis_unnorm(wafer_rgb)
    
    # Get attention maps
    attn_v2 = get_attention_maps(dinov2_model, img_tensor, DEVICE)
    attn_v3 = get_attention_maps(dinov3_model, img_tensor, DEVICE)
    
    # Plot: Original | DINOv2 attn | DINOv3 attn | Difference
    axes[row, 0].imshow(raw_img)
    axes[row, 0].set_title(f'{defect_class}' if row == 0 else defect_class, fontsize=12)
    if row == 0:
        axes[row, 0].set_title('Original Wafer Map', fontsize=12, fontweight='bold')
    
    im1 = axes[row, 1].imshow(attn_v2, cmap='inferno', interpolation='bilinear')
    if row == 0:
        axes[row, 1].set_title('DINOv2 Attention', fontsize=12, fontweight='bold')
    
    im2 = axes[row, 2].imshow(attn_v3, cmap='inferno', interpolation='bilinear')
    if row == 0:
        axes[row, 2].set_title('DINOv3 Attention', fontsize=12, fontweight='bold')
    
    # Difference map (DINOv3 - DINOv2)
    # Resize to match if needed
    from PIL import Image
    if attn_v2.shape != attn_v3.shape:
        attn_v2_resized = np.array(Image.fromarray(attn_v2).resize(
            (attn_v3.shape[1], attn_v3.shape[0]), Image.BILINEAR
        ))
    else:
        attn_v2_resized = attn_v2
    
    diff = attn_v3 - attn_v2_resized
    im3 = axes[row, 3].imshow(diff, cmap='RdBu_r', interpolation='bilinear',
                               vmin=-0.5, vmax=0.5)
    if row == 0:
        axes[row, 3].set_title('Δ (v3 − v2)', fontsize=12, fontweight='bold')
    
    # Add class label on left
    axes[row, 0].set_ylabel(defect_class, fontsize=12, fontweight='bold', rotation=0,
                            labelpad=60, va='center')
    
    for col in range(4):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

plt.suptitle('Attention Maps: Where Each Model Looks on Wafer Maps',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('attention_maps_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Bright regions = high attention. The difference column shows where")
print("DINOv3 attends MORE (red) or LESS (blue) than DINOv2.")
print("Ideally, attention should concentrate on the defect region, not the background.")

## 11. Feature Quality Analysis

Beyond classification accuracy, we can measure the intrinsic quality of the embedding spaces.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier

# Subsample for silhouette (expensive on full dataset)
N_SIL = 5000
sil_idx = np.random.choice(len(test_feats_v2), min(N_SIL, len(test_feats_v2)), replace=False)

# Silhouette score (higher = better cluster separation)
sil_v2 = silhouette_score(test_feats_v2[sil_idx], test_labels_arr[sil_idx], sample_size=N_SIL)
sil_v3 = silhouette_score(test_feats_v3[sil_idx], test_labels_arr[sil_idx], sample_size=N_SIL)

# k-NN accuracy (tests local structure of embedding space)
knn_v2 = KNeighborsClassifier(n_neighbors=20)
knn_v2.fit(train_feats_v2, train_labels_arr)
knn_acc_v2 = knn_v2.score(test_feats_v2, test_labels_arr)

knn_v3 = KNeighborsClassifier(n_neighbors=20)
knn_v3.fit(train_feats_v3, train_labels_arr)
knn_acc_v3 = knn_v3.score(test_feats_v3, test_labels_arr)

print("Feature Quality Metrics")
print("=" * 50)
print(f"{'Metric':<30} {'DINOv2':>10} {'DINOv3':>10}")
print("-" * 50)
print(f"{'Silhouette Score':<30} {sil_v2:>10.4f} {sil_v3:>10.4f}")
print(f"{'k-NN Accuracy (k=20)':<30} {knn_acc_v2:>10.4f} {knn_acc_v3:>10.4f}")
print(f"{'Linear Probe Accuracy':<30} {results_v2['test_acc']:>10.4f} {results_v3['test_acc']:>10.4f}")
print(f"{'Feature Extraction Speed':<30} {v2_train_time:>9.1f}s {v3_train_time:>9.1f}s")

In [ ]:
# Radar chart of all metrics
categories = ['Test Accuracy', 'F1 (Macro)', 'Silhouette', 'k-NN Accuracy', 'F1 (Weighted)']
v2_values = [
    results_v2['test_acc'], results_v2['test_f1_macro'],
    max(sil_v2, 0),  # Silhouette can be negative
    knn_acc_v2, results_v2['test_f1_weighted']
]
v3_values = [
    results_v3['test_acc'], results_v3['test_f1_macro'],
    max(sil_v3, 0),
    knn_acc_v3, results_v3['test_f1_weighted']
]

# Normalize silhouette to [0,1] range for fair comparison on radar
max_sil = max(max(sil_v2, 0), max(sil_v3, 0), 0.01)
v2_values[2] = v2_values[2] / max_sil
v3_values[2] = v3_values[2] / max_sil

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
v2_values_plot = v2_values + [v2_values[0]]
v3_values_plot = v3_values + [v3_values[0]]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.plot(angles, v2_values_plot, 'o-', linewidth=2, label='DINOv2', color='#4285F4')
ax.fill(angles, v2_values_plot, alpha=0.15, color='#4285F4')
ax.plot(angles, v3_values_plot, 's-', linewidth=2, label='DINOv3', color='#EA4335')
ax.fill(angles, v3_values_plot, alpha=0.15, color='#EA4335')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)

plt.tight_layout()
plt.savefig('radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary & Conclusions

In [ ]:
# Final summary table
print("\n" + "=" * 70)
print("         DINOV2 vs DINOV3 — WAFER MAP CLASSIFICATION SUMMARY")
print("=" * 70)
print(f"\n{'Metric':<35} {'DINOv2':>12} {'DINOv3':>12} {'Delta':>10}")
print("-" * 70)

comparisons = [
    ('Test Accuracy', results_v2['test_acc'], results_v3['test_acc']),
    ('F1 Score (Macro)', results_v2['test_f1_macro'], results_v3['test_f1_macro']),
    ('F1 Score (Weighted)', results_v2['test_f1_weighted'], results_v3['test_f1_weighted']),
    ('Silhouette Score', sil_v2, sil_v3),
    ('k-NN Accuracy (k=20)', knn_acc_v2, knn_acc_v3),
    ('Feature Extraction (sec)', v2_train_time + v2_test_time, v3_train_time + v3_test_time),
]

for name, v2, v3 in comparisons:
    delta = v3 - v2
    sign = '+' if delta > 0 else ''
    # For time, lower is better
    if 'sec' in name:
        indicator = '  ⬇' if delta < 0 else '  ⬆'
    else:
        indicator = '  ✓' if delta > 0 else '  ✗'
    print(f"{name:<35} {v2:>12.4f} {v3:>12.4f} {sign}{delta:>8.4f}{indicator}")

print("-" * 70)
print(f"\n{'Winner':>35}: ", end='')
if results_v3['test_acc'] > results_v2['test_acc']:
    print("DINOv3 ✓")
elif results_v3['test_acc'] < results_v2['test_acc']:
    print("DINOv2 ✓")
else:
    print("Tie")

print("\n" + "=" * 70)
print("NOTES:")
print("- Both models use generic pretrained weights (NOT domain-adapted)")
print("- Neither has seen wafer maps during pretraining")
print("- NV-DINOv2 (domain-adapted) would outperform both on this task")
print("- DINOv3's advantage should be larger on dense/spatial tasks")
print("  (die-level segmentation) vs whole-map classification shown here")
print("- Few-shot curves show which model needs fewer labels — critical")
print("  for real fab deployment where expert labeling is expensive")
print("=" * 70)

In [ ]:
# Save all results for later analysis
results_summary = {
    'dinov2': {
        'test_acc': results_v2['test_acc'],
        'test_f1_macro': results_v2['test_f1_macro'],
        'test_f1_weighted': results_v2['test_f1_weighted'],
        'silhouette': sil_v2,
        'knn_acc': knn_acc_v2,
        'extraction_time': v2_train_time + v2_test_time,
        'few_shot': fewshot_v2,
        'per_class_f1': {DEFECT_CLASSES[i]: f for i, f in enumerate(f1_per_class_v2)},
    },
    'dinov3': {
        'test_acc': results_v3['test_acc'],
        'test_f1_macro': results_v3['test_f1_macro'],
        'test_f1_weighted': results_v3['test_f1_weighted'],
        'silhouette': sil_v3,
        'knn_acc': knn_acc_v3,
        'extraction_time': v3_train_time + v3_test_time,
        'few_shot': fewshot_v3,
        'per_class_f1': {DEFECT_CLASSES[i]: f for i, f in enumerate(f1_per_class_v3)},
    },
}

with open('comparison_results.pkl', 'wb') as f:
    pickle.dump(results_summary, f)

print("Results saved to comparison_results.pkl")
print("\nGenerated visualization files:")
for f in sorted(Path('.').glob('*.png')):
    print(f"  📊 {f.name}")